# 我孫子市の天気を見える化！― 今日の空模様とこれからの3日間 ―

**制作形態：個人制作**  
**氏名：［ここに自分の氏名を入力］**

## Notebookの説明
千葉県我孫子市付近の天気情報をOpen-Meteo APIから取得し、現在の気温・湿度・風速と今後3日間の予報を表やグラフで確認するNotebookである。外出や服装を考えるときに役立つ情報を見える化する。実行時に最新データを取得するため、インターネット接続が必要。

## データの出典
- Open-Meteo Weather Forecast API
- ドキュメント：https://open-meteo.com/en/docs
- API：https://api.open-meteo.com/v1/forecast
- 対象地点：千葉県我孫子市付近（緯度35.8729、経度140.0104）
- タイムゾーン：Asia/Tokyo

データは気象モデルに基づく予報であり、観測所の実測値とは限らない。

## 1. ライブラリの読み込み

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.unicode_minus"] = False
print("ライブラリの読み込みが完了しました。")

## 2. 我孫子市の天気データを取得

In [ ]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 35.8729,
    "longitude": 140.0104,
    "current": ["temperature_2m", "relative_humidity_2m", "apparent_temperature",
                "precipitation", "weather_code", "wind_speed_10m"],
    "hourly": ["temperature_2m", "precipitation_probability", "precipitation", "wind_speed_10m"],
    "daily": ["weather_code", "temperature_2m_max", "temperature_2m_min", "precipitation_sum"],
    "forecast_days": 3,
    "timezone": "Asia/Tokyo"
}
response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
data = response.json()
print("データ取得成功")
print("地点：我孫子市付近")
print("タイムゾーン：", data.get("timezone"))
print("データ時刻：", data.get("current", {}).get("time"))

## 3. 現在の天気を確認

In [ ]:
weather_names = {
    0:"快晴", 1:"おおむね晴れ", 2:"一部曇り", 3:"曇り",
    45:"霧", 48:"着氷性の霧", 51:"弱い霧雨", 53:"霧雨", 55:"強い霧雨",
    61:"弱い雨", 63:"雨", 65:"強い雨", 71:"弱い雪", 73:"雪", 75:"強い雪",
    80:"にわか雨", 81:"強いにわか雨", 82:"激しいにわか雨",
    95:"雷雨", 96:"ひょうを伴う雷雨", 99:"強いひょうを伴う雷雨"
}
current = data["current"]
code = int(current["weather_code"])
current_table = pd.DataFrame({
    "項目":["天気","気温","体感温度","相対湿度","降水量","風速"],
    "値":[weather_names.get(code, f"天気コード {code}"),
          f'{current["temperature_2m"]} ℃', f'{current["apparent_temperature"]} ℃',
          f'{current["relative_humidity_2m"]} %', f'{current["precipitation"]} mm',
          f'{current["wind_speed_10m"]} km/h']
})
display(current_table)

## 4. 今後3日間の天気予報

In [ ]:
daily = data["daily"]
forecast = pd.DataFrame({
    "日付":daily["time"],
    "天気":[weather_names.get(int(x), f"コード{x}") for x in daily["weather_code"]],
    "最高気温 (℃)":daily["temperature_2m_max"],
    "最低気温 (℃)":daily["temperature_2m_min"],
    "降水量 (mm)":daily["precipitation_sum"]
})
display(forecast)

## 5. 時間ごとの気温変化をグラフで表示

In [ ]:
hourly = data["hourly"]
hourly_df = pd.DataFrame({
    "日時":pd.to_datetime(hourly["time"]),
    "気温 (℃)":hourly["temperature_2m"],
    "降水確率 (%)":hourly["precipitation_probability"],
    "風速 (km/h)":hourly["wind_speed_10m"]
})
plt.figure()
plt.plot(hourly_df["日時"], hourly_df["気温 (℃)"], marker="o", markersize=3)
plt.title("我孫子市付近の時間別気温予報")
plt.xlabel("日時")
plt.ylabel("気温 (℃)")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

## 6. 予報から分かったことを自動でまとめる

In [ ]:
max_row = forecast.loc[forecast["最高気温 (℃)"].idxmax()]
min_row = forecast.loc[forecast["最低気温 (℃)"].idxmin()]
rain_total = forecast["降水量 (mm)"].sum()
print(f'3日間で最も高い最高気温：{max_row["日付"]} の {max_row["最高気温 (℃)"]} ℃')
print(f'3日間で最も低い最低気温：{min_row["日付"]} の {min_row["最低気温 (℃)"]} ℃')
print(f'3日間の予報降水量の合計：{rain_total:.1f} mm')
if forecast["降水量 (mm)"].max() >= 5:
    print("まとまった雨が予報されている日があります。外出前に最新予報を確認してください。")
else:
    print("予報上、まとまった降水量の日は多くありません。")

## 7. 自分で追加した機能・Notebookの特徴
- 我孫子市付近の地点を緯度・経度で指定して天気を取得する。
- 気温だけでなく体感温度・湿度・風速・降水量も表示する。
- 3日間の予報を表にまとめ、時間別の気温変化をグラフ化する。
- 最高気温・最低気温・降水量合計をPythonで自動計算する。
- 実行時にAPIからデータを取得するため、新しい予報を確認できる。

## 8. これから改善したいこと・追加したい機能
- 7日間の週間予報に対応させたい。
- 降水確率や風速もグラフ化し、傘や服装を選びやすくしたい。
- 我孫子市内の複数地点を比較したい。
- APIに接続できない場合のエラー表示や再試行機能を追加したい。
- 天気コードの説明を増やし、より分かりやすく表示したい。

## 9. まとめ・注意点
このNotebookでは、Open-Meteo APIから我孫子市付近の天気予報を取得し、現在の天気、3日間の予報、時間別気温のグラフを作成した。表とグラフを組み合わせることで、気温の変化を視覚的に確認できる。

ただし、予報は気象モデルに基づくため実際の天候と異なる場合がある。また、指定地点は我孫子市付近の代表座標であり、市内すべての場所の天気を示すものではない。防災や外出の判断には気象庁などの最新情報も確認する。

**提出前：** 氏名を記入し、全Cellを上から実行して出力を表示した状態で保存する。追加ファイルは不要。インターネット接続が必要。